# 1. Setup

**Approach 2 · Cross-Crop Feature Recombination**

Compare A2, F1 and F2 in Development, then refit and officially test the selected recipe. Run from the team project using Python 3.13.17 and the dependencies in `requirements.txt`. Set `DEBUG=False` and use **Run All**. Debug runs a short training-only check.

In [ ]:
DEBUG = False

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import random
import sys

import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
BATCH_SIZE = 8 if DEBUG else 16
EPOCHS = 1 if DEBUG else 20
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 0
RUN_NAME = "approach2_crosscrop_recombination_selected" + ("_debug" if DEBUG else "")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
DEVELOPMENT_DIR = OUTPUT_DIR / "development"
FINAL_DIR = OUTPUT_DIR / "final"
OFFICIAL_DIR = OUTPUT_DIR / "official" / "pair_identity_order"
CHECKPOINT_DIR = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID

HEAD_LR, WEIGHT_DECAY = 1e-4, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.1, 0.1
GAMMAS = [step / 2 for step in range(25)]
PROJECTION_DIM = 128
FUSION_WEIGHT, PAIR_LOSS_WEIGHT = 1.0, 0.5
FUSION_DIAGNOSTIC_WEIGHTS = (0.25, 0.5, 1.0, 2.0)
RECIPES = {
    "A2_Control": {"fusion": False, "swap": False},
    "F1_RealFusion": {"fusion": True, "swap": False},
    "F2_CrossCropRecombination": {"fusion": True, "swap": True},
}
COMPARISON_BASES = {
    "F1_RealFusion": ("A2_Control",),
    "F2_CrossCropRecombination": ("F1_RealFusion", "A2_Control"),
}

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Missing training metadata or images under {team.DATA_DIR}.")
if (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) != (323, 10, 333):
    raise ValueError("Use the team's 323/333 common files.")
print(f"{DEVICE} | FP32 | batch {BATCH_SIZE} | {EPOCHS} epochs | DEBUG={DEBUG}")


# 2. Data

Use only `train.csv` for development. Hold out every image of ten training-derived combinations while preserving each crop and disease concept in the fit subset. Five combinations and a seen validation subset guide selection. The other five and a separate seen validation subset provide an independent check. Batch matching uses the fit subset. Full refit puts all training-derived combinations back into training.

In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
_, image_transform = get_transforms("imagenet")


def calibration_split(fit_frame, val_frame):
    """Hold out complete training pairs from the development training."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Calibration needs both seen and proxy images in each group.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "head_lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW", "schedule": "CosineAnnealingLR", "label_smoothing": SMOOTHING,
    "decorrelation_weight": DECOR_WEIGHT, "supcon_weight": SUPCON_WEIGHT, "temperature": TEMPERATURE,
    "projection_dim": PROJECTION_DIM, "fusion_weight": FUSION_WEIGHT, "pair_loss_weight": PAIR_LOSS_WEIGHT,
    "recipes": RECIPES, "fusion_diagnostic_weights": list(FUSION_DIAGNOSTIC_WEIGHTS), "calibration_grid": GAMMAS, "train_csv_sha256": sha256_file(team.TRAIN_CSV),
    "python": sys.version.split()[0], "torch": str(torch.__version__), "timm": timm.__version__, "debug": DEBUG,
}


# 3. Model

A2 uses crop and disease cross-entropy, logit Decorrelation and Cross-crop SupCon. F1 and F2 add separately supervised crop and disease projections. A learned fusion head compares the joint representation with a prototype composed from the candidate's crop and disease embeddings. Each pair score adds this bounded cosine compatibility to the two concept log-probabilities. The prototype is computed by the same function for every candidate, including novel combinations.

For a training anchor, F2 keeps its crop features and takes disease features from a different crop with the same disease label. The labelled target remains the anchor's known crop–disease training pair. For eligible anchors, average the real and swapped pair loss so that F1 and F2 keep the same overall pair-loss weight. Each image's original concept losses are retained. The swap uses latent tensors and the fit subset's pair table.

Separating crop and disease representations and training on feature recombinations adapts the idea in [Pairwise Feature Learning for Unseen Plant Disease Recognition, ICIP 2023](https://ir.unimas.my/id/eprint/43065/2/PAIRWISE.pdf). The three-arm experiment tests that idea on DPD-Small. The original PlantVillage experiment used three unseen pairs and a different prediction protocol. [Full DPD benchmark, ACM MM 2025](https://agritrop.cirad.fr/618258/1/618258.pdf#page=5)

F2 swaps a representation between two fit images that share a disease. The supervised target is the anchor's observed training pair. This tests whether disease evidence transfers across crops without assigning an unseen pair label during training.

In [ ]:
class MultiTaskNet(nn.Module):
    """Latest Baseline 2 architecture."""
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.crop_head = nn.Linear(self.backbone.num_features, NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


def pair_scores(crop_logits, disease_logits, pairs):
    pairs = pairs.to(crop_logits.device)
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()


class PairFusionNet(MultiTaskNet):
    """Predict crop, disease, and a residual compatibility for each candidate pair."""

    def __init__(self, pretrained):
        super().__init__(pretrained=pretrained)
        width = self.backbone.num_features
        self.crop_projection = nn.Sequential(nn.Linear(width, PROJECTION_DIM), nn.LayerNorm(PROJECTION_DIM))
        self.disease_projection = nn.Sequential(nn.Linear(width, PROJECTION_DIM), nn.LayerNorm(PROJECTION_DIM))
        self.crop_projected_head = nn.Linear(PROJECTION_DIM, NUM_CROPS, bias=False)
        self.disease_projected_head = nn.Linear(PROJECTION_DIM, NUM_DISEASES, bias=False)
        nn.init.zeros_(self.crop_projected_head.weight)
        nn.init.zeros_(self.disease_projected_head.weight)
        self.fusion = nn.Sequential(nn.Linear(2 * PROJECTION_DIM, PROJECTION_DIM), nn.GELU(),
                                    nn.Linear(PROJECTION_DIM, PROJECTION_DIM))
        self.crop_prototypes = nn.Embedding(NUM_CROPS, PROJECTION_DIM)
        self.disease_prototypes = nn.Embedding(NUM_DISEASES, PROJECTION_DIM)

    def encode(self, images):
        base = self.backbone(images)
        crop = self.crop_projection(base)
        disease = self.disease_projection(base)
        crop_logits = self.crop_head(base) + self.crop_projected_head(crop)
        disease_logits = self.disease_head(base) + self.disease_projected_head(disease)
        return crop_logits, disease_logits, crop, disease

    def forward(self, images):
        crop_logits, disease_logits, _, _ = self.encode(images)
        return crop_logits, disease_logits

    def fusion_scores(self, crop, disease, pairs, prototype_bank=None):
        pairs = pairs.to(crop.device)
        joint = F.normalize(self.fusion(torch.cat((crop, disease), dim=1)), dim=1)
        if prototype_bank is None:
            prototype_bank = (self.crop_prototypes.weight, self.disease_prototypes.weight)
        crop_bank, disease_bank = prototype_bank
        prototypes = F.normalize(crop_bank[pairs[:, 0]] * disease_bank[pairs[:, 1]], dim=1)
        return joint @ prototypes.T

    def score_components(self, crop_logits, disease_logits, crop, disease, pairs):
        return pair_scores(crop_logits, disease_logits, pairs) + FUSION_WEIGHT * self.fusion_scores(crop, disease, pairs)


def build_network(spec, pretrained):
    network = PairFusionNet if spec["fusion"] else MultiTaskNet
    return network(pretrained=pretrained)


def score_images(model, images, pairs):
    if isinstance(model, PairFusionNet):
        components = model.encode(images)
        return model.score_components(*components, pairs), components[:2]
    logits = model(images)
    return pair_scores(*logits, pairs), logits


class CrossCropBatchSampler:
    """Batch eligible cross-crop pairs together while visiting each fit image once."""

    def __init__(self, frame, batch_size):
        self.crops = frame.crop.tolist()
        self.diseases = frame.disease.tolist()
        self.by_disease = frame.groupby("disease").indices
        self.batch_size = batch_size
        self.epoch = 0

    def __len__(self):
        return (len(self.crops) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        rng = random.Random(team.SEED + self.epoch)
        self.epoch += 1
        order = rng.sample(range(len(self.crops)), k=len(self.crops))
        remaining, batch = set(order), []
        for index in order:
            if index not in remaining:
                continue
            remaining.remove(index)
            batch.append(index)
            if len(batch) < self.batch_size:
                eligible = [other for other in self.by_disease[self.diseases[index]]
                            if other in remaining and self.crops[other] != self.crops[index]]
                if eligible:
                    donor = int(rng.choice(eligible))
                    remaining.remove(donor)
                    batch.append(donor)
            if len(batch) == self.batch_size:
                yield batch
                batch = []
        if batch:
            yield batch


def cross_crop_sources(crop, disease):
    eligible = disease[:, None].eq(disease[None, :]) & crop[:, None].ne(crop[None, :])
    rows = eligible.any(dim=1).nonzero(as_tuple=True)[0]
    donors = eligible[rows].to(torch.int64).argmax(dim=1)
    return rows, donors


def make_fit_loader(frame, transform):
    dataset = PlantDataset(frame, transform, "imagenet")
    return DataLoader(dataset, batch_sampler=CrossCropBatchSampler(dataset.frame, BATCH_SIZE),
                      num_workers=team.NUM_WORKERS, generator=torch.Generator().manual_seed(team.SEED))


def training_objective(spec, fit):
    fit_pairs = target_lookup = None
    if spec["fusion"]:
        fit_ids = torch.tensor(sorted(fit.pair_row.unique()), dtype=torch.long)
        fit_pairs = train_pairs[fit_ids].to(DEVICE)
        target_lookup = torch.full((len(train_pairs),), -1, dtype=torch.long)
        target_lookup[fit_ids] = torch.arange(len(fit_ids))
        target_lookup = target_lookup.to(DEVICE)

    def objective(model, criterion, images, crop, disease, pair_rows):
        if spec["fusion"]:
            components = model.encode(images)
            crop_logits, disease_logits = components[:2]
        else:
            crop_logits, disease_logits = model(images)

        disease_repr = center_logits(disease_logits)
        contrast, supcon_anchors = cross_crop_supcon(disease_repr, crop, disease)
        loss = (criterion(crop_logits, crop) + criterion(disease_logits, disease)
                + DECOR_WEIGHT * decorrelation(center_logits(crop_logits), disease_repr)
                + SUPCON_WEIGHT * contrast)
        swaps = 0
        if spec["fusion"]:
            targets = target_lookup[pair_rows]
            # Every fit target uses the same train-only candidate table for both arms.
            per_image = F.cross_entropy(model.score_components(*components, fit_pairs), targets, reduction="none")
            pair_loss = per_image.sum()
            if spec["swap"]:
                anchors, donors = cross_crop_sources(crop, disease)
                swaps = len(anchors)
                if swaps:
                    c_logits, d_logits, c_features, d_features = components
                    swapped = model.score_components(c_logits[anchors], d_logits[donors],
                                                     c_features[anchors], d_features[donors], fit_pairs)
                    swap_loss = F.cross_entropy(swapped, targets[anchors], reduction="sum")
                    pair_loss = pair_loss + 0.5 * (swap_loss - per_image[anchors].sum())
            loss = loss + PAIR_LOSS_WEIGHT * pair_loss / len(images)
        return loss, supcon_anchors, swaps

    return objective


# 4. Training

Train A2, F1 and F2 on the shared training-derived fit split. Select each recipe's epoch and one winning recipe by raw tune harmonic mean. Tune calibration separately on the same training-derived split and report the independent proxy check. Use this check for ablation comparisons. Its scores do not determine the winner.

Review the development comparison under `development/`. Selection is fixed before stage 5.

In [ ]:
def make_model(spec):
    return build_network(spec, pretrained=True).to(DEVICE).eval()


def make_loader(frame, transform, shuffle):
    return DataLoader(PlantDataset(frame, transform, "imagenet"), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=team.NUM_WORKERS, drop_last=False,
                      generator=torch.Generator().manual_seed(team.SEED))


def train_epoch(model, optimizer, criterion, loader, objective, recipe, stage, epoch, epochs):
    loss_total, samples, valid_anchors, swapped_anchors = 0., 0, 0, 0
    for step, (images, crop, disease, pair_rows) in enumerate(loader, 1):
        images, crop, disease = (batch.to(DEVICE) for batch in (images, crop, disease))
        loss, anchors, swaps = objective(model, criterion, images, crop, disease, pair_rows.to(DEVICE))
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss in {recipe}.")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        loss_total += loss.item() * len(images)
        samples += len(images)
        valid_anchors += int(anchors)
        swapped_anchors += int(swaps)
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{recipe} {stage} epoch {epoch}/{epochs} · batch {step}/{len(loader)}", flush=True)
    if samples != len(loader.dataset):
        raise RuntimeError("The fit loader did not process all intended images.")
    return {"epoch": epoch, "images_trained": samples,
            "train_loss": loss_total / samples, "supcon_anchor_fraction": valid_anchors / samples,
            "swapped_anchor_fraction": swapped_anchors / samples}


@torch.no_grad()
def development_scores(model, loader):
    scores, labels = [], []
    for images, _, _, target in loader:
        batch_scores, _ = score_images(model, images.to(DEVICE), train_pairs)
        scores.append(batch_scores.cpu())
        labels.append(target)
    return torch.cat(scores), torch.cat(labels)


def calibration_metrics(seen, proxy, fit_mask, gamma):
    seen_scores, y_seen = seen
    proxy_scores, y_proxy = proxy
    seen_scores = seen_scores - gamma * fit_mask
    proxy_scores = proxy_scores - gamma * fit_mask
    seen_top1 = top_k_accuracy(seen_scores, y_seen, 1)
    proxy_top1 = top_k_accuracy(proxy_scores, y_proxy, 1)
    return {"gamma": gamma, "seen_top1": seen_top1, "proxy_top1": proxy_top1,
            "seen_top5": top_k_accuracy(seen_scores, y_seen, 5),
            "proxy_top5": top_k_accuracy(proxy_scores, y_proxy, 5),
            "harmonic_top1": harmonic_mean(seen_top1, proxy_top1)}


def selection_key(row):
    return row["harmonic_top1"], row["proxy_top1"], row["seen_top1"], -row["epoch"]


def snapshot_trainable(model):
    return {name: param.detach().cpu().clone()
            for name, param in model.named_parameters() if param.requires_grad}


@torch.no_grad()
def restore_trainable(model, state):
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.copy_(state[name])


@torch.no_grad()
def compute_mean_prototypes(model, fit_loader):
    """Calculate normalized class means from projected fit images only."""
    sums = [torch.zeros(n, PROJECTION_DIM, device=DEVICE) for n in (NUM_CROPS, NUM_DISEASES)]
    counts = [torch.zeros(n, device=DEVICE) for n in (NUM_CROPS, NUM_DISEASES)]
    for images, crop, disease, _ in fit_loader:
        _, _, crop_features, disease_features = model.encode(images.to(DEVICE))
        for total, count, labels, features in zip(sums, counts, (crop, disease), (crop_features, disease_features)):
            labels = labels.to(DEVICE)
            total.index_add_(0, labels, F.normalize(features, dim=1))
            count.index_add_(0, labels, torch.ones_like(labels, dtype=count.dtype))
    for index, count in enumerate(counts):
        required = train_pairs[:, index].unique().to(DEVICE)
        if (count[required] == 0).any():
            raise RuntimeError("A development candidate concept has no fit prototype.")
    return tuple(F.normalize(total / count.clamp_min(1)[:, None], dim=1) for total, count in zip(sums, counts))


@torch.no_grad()
def fusion_diagnostic_scores(model, loader, mean_bank=None):
    items = []
    for images, _, _, targets in loader:
        crop, disease, crop_features, disease_features = model.encode(images.to(DEVICE))
        additive = pair_scores(crop, disease, train_pairs)
        learned = model.fusion_scores(crop_features, disease_features, train_pairs)
        batch = [additive, learned, targets]
        if mean_bank is not None:
            batch.append(model.fusion_scores(crop_features, disease_features, train_pairs, mean_bank))
        items.append(batch)
    return tuple(torch.cat([tensor.cpu() for tensor in column]) for column in zip(*items))


def run_fusion_diagnostics(model, spec, fit, groups, fit_mask, output):
    # These diagnostics cannot change selection, gamma, or the locked official recipes.
    mean_bank = None
    if spec["swap"]:
        mean_bank = compute_mean_prototypes(model, make_loader(fit, image_transform, shuffle=False))
    tune = [fusion_diagnostic_scores(model, groups[key], mean_bank) for key in ("tune_seen", "tune_proxy")]
    rows = []
    for weight in FUSION_DIAGNOSTIC_WEIGHTS:
        scored = [(base + weight * learned, labels) for base, learned, labels, *_ in tune]
        rows.append({"fusion_weight": weight, "main_weight": weight == FUSION_WEIGHT,
                     **calibration_metrics(*scored, fit_mask, 0.)})
    pd.DataFrame(rows).to_csv(output / "fusion_weight_sensitivity.csv", index=False)

    if mean_bank is not None:
        check = [fusion_diagnostic_scores(model, groups[key], mean_bank) for key in ("check_seen", "check_proxy")]
        rows = []
        for split, caches in (("tune", tune), ("check", check)):
            for method, score_index in (("learned", 1), ("fit_mean_replacement", 3)):
                scored = [(cache[0] + FUSION_WEIGHT * cache[score_index], cache[2]) for cache in caches]
                rows.append({"split": split, "prototype_source": method, "fusion_weight": FUSION_WEIGHT,
                             **calibration_metrics(*scored, fit_mask, 0.)})
        pd.DataFrame(rows).to_csv(output / "prototype_ablation.csv", index=False)


def develop_recipe(recipe, spec, fit, fit_transform, groups, fit_mask):
    team.set_seed()
    model = make_model(spec)
    team.set_seed()
    fit_loader = make_fit_loader(fit, fit_transform)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                  lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    objective = training_objective(spec, fit)
    history, best, best_weights, best_scores = [], None, None, None
    for epoch in range(1, EPOCHS + 1):
        row = {"recipe": recipe, "feature_swapping": spec["swap"],
               **train_epoch(model, optimizer, criterion, fit_loader, objective, recipe, "development", epoch, EPOCHS)}
        scheduler.step()
        if not DEBUG:
            tune = [development_scores(model, groups[key]) for key in ("tune_seen", "tune_proxy")]
            row.update(calibration_metrics(*tune, fit_mask, 0.))
            if best is None or selection_key(row) > selection_key(best):
                best = row.copy()
                best_weights = snapshot_trainable(model)
                best_scores = tune
        history.append(row)
        print(row, flush=True)

    output = DEVELOPMENT_DIR / recipe
    output.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(output / "training_history.csv", index=False)
    if DEBUG:
        return None
    if spec["swap"] and not any(row["swapped_anchor_fraction"] > 0 for row in history):
        raise RuntimeError("Feature recombination requires fit images sharing a disease across crops.")

    grid = [calibration_metrics(*best_scores, fit_mask, gamma) for gamma in GAMMAS]
    gamma = max(grid, key=lambda row: (row["harmonic_top1"], row["proxy_top1"],
                                       row["seen_top1"], -row["gamma"]))["gamma"]
    restore_trainable(model, best_weights)
    if spec["fusion"]:
        run_fusion_diagnostics(model, spec, fit, groups, fit_mask, output)
    check = [development_scores(model, groups[key]) for key in ("check_seen", "check_proxy")]
    check_rows = [{"recipe": recipe, "scoring": rule, **calibration_metrics(*check, fit_mask, penalty)}
                  for rule, penalty in (("raw", 0.), ("calibrated", gamma))]
    pd.DataFrame(grid).to_csv(output / "calibration_tune_grid.csv", index=False)
    pd.DataFrame(check_rows).to_csv(output / "calibration_check.csv", index=False)
    return {"selection": best, "gamma": gamma, "independent_check": check_rows,
            "trainable_parameter_count": sum(p.numel() for p in model.parameters() if p.requires_grad)}


def run_development():
    team.set_seed()
    train, val, seen, unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if seen is not None or unseen is not None:
        raise RuntimeError("Development loaders must use the training CSV only.")
    fit = train.dataset.frame
    fit_mask, frames, groups = None, {}, {}
    if not DEBUG:
        fit_mask, frames = calibration_split(fit, val.dataset.frame)
        allowed = fit_mask.nonzero(as_tuple=True)[0].tolist()
        fit = fit.loc[fit.pair_row.isin(allowed)]
        groups = {key: make_loader(frame, image_transform, shuffle=False) for key, frame in frames.items()}

    selections = {recipe: develop_recipe(recipe, spec, fit, train.dataset.transform, groups, fit_mask)
                  for recipe, spec in RECIPES.items()}
    if DEBUG:
        return {}
    # Independent check results are reported only. The winner uses raw tune metrics.
    chosen = max(selections, key=lambda recipe: selection_key(selections[recipe]["selection"]))
    plan = {"recipe_selected_on_tune": chosen, "recipes": selections,
            "tune_pair_rows": sorted(frames["tune_proxy"].pair_row.unique().tolist()),
            "check_pair_rows": sorted(frames["check_proxy"].pair_row.unique().tolist())}
    atomic_json(DEVELOPMENT_DIR / "selection_plan.json", plan)
    pd.DataFrame([{**value["selection"], "locked_gamma": value["gamma"], "selected_on_tune": recipe == chosen}
                  for recipe, value in selections.items()]).to_csv(
        DEVELOPMENT_DIR / "selected_models.csv", index=False)
    check_rows = [
        {"recipe": recipe, "selected_on_tune": recipe == chosen, **row}
        for recipe, selection in selections.items() for row in selection["independent_check"]
    ]
    summary = pd.DataFrame(check_rows)
    summary.to_csv(DEVELOPMENT_DIR / "development_ablation.csv", index=False)
    print(f"Selected on raw tune harmonic mean: {chosen}", flush=True)
    display(pd.read_csv(DEVELOPMENT_DIR / "selected_models.csv"))
    display(summary)
    return plan


plan = run_development()


# 5. Final evaluation

Refit the recipe selected in section 4 for its selected epoch count using all rows in `train.csv`. Save the full checkpoint and lock the chosen recipe, epoch, and calibration penalty. The independent proxy check above describes the development checkpoints. Its held-out pairs are included in the final refit.

The official test CSVs are first opened in section 6.

In [ ]:
def read_final_lock(recipe):
    path = FINAL_DIR / recipe / "selection_lock.json"
    checkpoint = CHECKPOINT_DIR / recipe / "model.pt"
    lock = json.loads(path.read_text(encoding="utf-8"))
    if (lock["settings"] != settings or lock["recipe"] != recipe
            or lock["selection_plan"] != plan or lock["seen_pairs"] != train_pairs.tolist()
            or lock["checkpoint_sha256"] != sha256_file(checkpoint)):
        raise RuntimeError(f"Final checkpoint or plan changed for {recipe}.")
    return lock


def refit_recipe(recipe, spec, selection):
    team.set_seed()
    model = make_model(spec)
    train_transform, _ = get_transforms("imagenet")
    team.set_seed()
    loader = make_fit_loader(train_full, train_transform)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                  lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    objective = training_objective(spec, train_full)
    epochs = selection["selection"]["epoch"]
    if not 1 <= epochs <= EPOCHS:
        raise ValueError(f"Invalid selected epoch for {recipe}.")
    history = []
    for epoch in range(1, epochs + 1):
        row = train_epoch(model, optimizer, criterion, loader, objective, recipe, "full refit", epoch, epochs)
        scheduler.step()
        history.append(row)
        print(row, flush=True)

    if spec["swap"] and not any(row["swapped_anchor_fraction"] > 0 for row in history):
        raise RuntimeError("The recombination refit did not find any eligible pairs.")
    checkpoint = CHECKPOINT_DIR / recipe / "model.pt"
    folder = FINAL_DIR / recipe
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    folder.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(folder / "refit_history.csv", index=False)
    refit = {"initialization": "original_pretrained_dinov2_fresh_heads",
             "data": "all_training_csv_rows", "images_per_epoch": len(train_full),
             "epochs": epochs, "cosine_t_max": EPOCHS}
    metadata = {"settings": settings, "recipe": recipe, "feature_swapping": spec["swap"],
                "seen_pairs": train_pairs.tolist(), "selection": selection["selection"],
                "gamma": selection["gamma"], "refit": refit}
    atomic_save(checkpoint, {"state_dict": model.cpu().state_dict(), **metadata})
    lock = {**metadata, "selection_plan": plan, "checkpoint_sha256": sha256_file(checkpoint),
            "checkpoint_bytes": checkpoint.stat().st_size}
    atomic_json(folder / "selection_lock.json", lock)
    return lock


def refit_selected():
    if sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]:
        raise RuntimeError("Training CSV changed after development.")
    if json.loads((DEVELOPMENT_DIR / "selection_plan.json").read_text(encoding="utf-8")) != plan:
        raise RuntimeError("The development plan changed after selection.")
    recipe = plan["recipe_selected_on_tune"]
    selection = plan["recipes"][recipe]
    print(f"Full refit: {recipe} for {selection['selection']['epoch']} epochs", flush=True)
    if (FINAL_DIR / recipe / "selection_lock.json").is_file():
        return read_final_lock(recipe)
    return refit_recipe(recipe, RECIPES[recipe], selection)


winner_run = None if DEBUG else refit_selected()


# 6. Official test

Verify and reload the selected checkpoint. Combine the 323 train-derived candidates with ten official unseen pair identities. The loader uses filenames and images. Attach per-image Ground Truth after both raw and calibrated predictions have been made.

The official table describes the selected model. Rank both test splits over all 333 candidates and report seen, unseen and overall Top-1 and Top-5 plus harmonic mean. Raw at gamma 0 is the main result and the calibrated rule is supplementary.

In [ ]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, perturb=None):
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, generator=torch.Generator().manual_seed(team.SEED))
    mean, std = (torch.tensor(v).view(1, 3, 1, 1) for v in team.NORM_STATS["imagenet"])
    parts, concepts = [], []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        scores, (crop_logits, disease_logits) = score_images(model, images.to(DEVICE), pairs)
        scores = scores.cpu()
        if perturb is None:
            concepts.append(torch.stack((crop_logits.argmax(1), disease_logits.argmax(1)), dim=1).cpu())
        if scores.shape != (len(images), len(pairs)) or not torch.isfinite(scores).all():
            raise ValueError("Unexpected official candidate predictions.")
        parts.append(scores)
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return torch.cat(parts), torch.cat(concepts) if concepts else None



def predict_recipe(recipe, run, pairs, filenames):
    saved = torch.load(CHECKPOINT_DIR / recipe / "model.pt", map_location="cpu", weights_only=True)
    for key in ("settings", "recipe", "feature_swapping", "selection", "gamma", "seen_pairs", "refit"):
        if saved[key] != run[key]:
            raise RuntimeError(f"Checkpoint and selection lock differ for {recipe}: {key}.")
    model = build_network(RECIPES[recipe], pretrained=False)
    model.load_state_dict(saved["state_dict"], strict=True)
    model = model.to(DEVICE).eval()
    boundary = len(train_pairs)
    membership = torch.arange(len(pairs)) < boundary
    clean, heads = zip(*(predict_official(model, pairs, names, split)
                         for names, split in zip(filenames, ("seen", "unseen"))))
    perturbed = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for condition, transform in PERTURBATIONS.items():
            perturbed[condition], _ = predict_official(model, pairs, filenames[0], condition, transform)

    predictions = {}
    for rule, gamma in (("raw", 0.0), ("calibrated", run["gamma"])):
        seen_scores, unseen_scores = (scores - gamma * membership for scores in clean)
        predictions[rule] = {
            "gamma": gamma,
            "seen": top_predictions(seen_scores),
            "unseen": top_predictions(unseen_scores),
            "seen_only": top_predictions(seen_scores[:, :boundary]),
            "unseen_only": top_predictions(unseen_scores[:, boundary:], offset=boundary),
            "robust": {name: (scores - gamma * membership).topk(1, dim=1).indices
                       for name, scores in perturbed.items()},
        }

    return predictions, clean, heads


def score_trial(run, pairs, forward_pass, y_seen, y_unseen):
    predictions, clean, (seen_heads, unseen_heads) = forward_pass
    boundary = len(train_pairs)
    y_all = torch.cat((y_seen, y_unseen))

    reports, ranking, per_pair, robustness, restricted = [], [], [], [], []
    # Rows are unseen images and columns are candidate pairs.
    true_crop = pairs[y_unseen, 0]
    same_crop_candidates = true_crop[:, None].eq(pairs[None, :, 0])
    seen_head_hits = seen_heads.eq(pairs[y_seen])
    unseen_head_hits = unseen_heads.eq(pairs[y_unseen])
    head_top1 = {
        f"{split}_{head}_evidence_top1": round(100 * matches[:, col].float().mean().item(), 2)
        for split, matches in (("seen", seen_head_hits), ("unseen", unseen_head_hits))
        for col, head in enumerate(("crop", "disease"))
    }
    for name, prediction in predictions.items():
        gamma = prediction["gamma"]
        raw_scores = clean[1]
        crop_oracle = raw_scores.masked_fill(~same_crop_candidates, -torch.inf).topk(1, dim=1).indices
        raw_margins = raw_scores.gather(1, y_unseen[:, None]).squeeze(1) - raw_scores[:, :boundary].amax(dim=1)
        final_margins = raw_margins + gamma
        only_correct = prediction["unseen_only"][1].squeeze(1).eq(y_unseen)
        metrics = {"scoring": name, "gamma": gamma, "feature_swapping": run["feature_swapping"],
                   "selected_on_tune": run["recipe"] == plan["recipe_selected_on_tune"],
                   "epoch": run["selection"]["epoch"], "candidates": len(pairs), **head_top1,
                   "raw_true_crop_candidate_top1": prediction_accuracy(crop_oracle, y_unseen)}
        for k in (1, 5):
            metrics[f"seen_top{k}"] = prediction_accuracy(prediction["seen"][k], y_seen)
            metrics[f"unseen_top{k}"] = prediction_accuracy(prediction["unseen"][k], y_unseen)
            joined = torch.cat((prediction["seen"][k], prediction["unseen"][k]))
            metrics[f"overall_top{k}"] = prediction_accuracy(joined, y_all)
        metrics["harmonic_mean"] = harmonic_mean(metrics["seen_top1"], metrics["unseen_top1"])
        metrics["abs_seen_unseen_gap_pp"] = round(abs(metrics["seen_top1"] - metrics["unseen_top1"]), 2)
        seen_winner, unseen_winner = (prediction[split][1].squeeze(1) for split in ("seen", "unseen"))
        metrics["seen_predicted_unseen"] = int((seen_winner >= boundary).sum())
        for split, winner, target in (("seen", seen_winner, y_seen), ("unseen", unseen_winner, y_unseen)):
            concept_hits = pairs[winner].eq(pairs[target]).float().mean(dim=0) * 100
            metrics[f"{split}_test_crop_acc"], metrics[f"{split}_test_disease_acc"] = [
                round(float(value), 2) for value in concept_hits]
        reports.append(metrics)

        correct = unseen_winner.eq(y_unseen)
        predicts_seen = unseen_winner < boundary
        blocked = only_correct & predicts_seen
        same_crop = pairs[unseen_winner, 0].eq(pairs[y_unseen, 0])
        ranking.append({"scoring": name, "unseen_images": len(y_unseen), "joint_correct": int(correct.sum()),
                        "unseen_only_correct": int(only_correct.sum()), "unseen_predicted_seen": int(predicts_seen.sum()),
                        "blocked_by_seen": int(blocked.sum()), "blocked_same_crop": int((blocked & same_crop).sum()),
                        "blocked_other_crop": int((blocked & ~same_crop).sum()),
                        "median_true_pair_minus_top_seen_score": float(final_margins.median())})
        for index in range(boundary, len(pairs)):
            mask = y_unseen.eq(index)
            crop, disease = pairs[index].tolist()
            per_pair.append({"scoring": name, "crop": crop, "disease": disease, "support": int(mask.sum()),
                             "joint_correct": int((mask & correct).sum()),
                             "unseen_only_correct": int((mask & only_correct).sum()),
                             "blocked_by_seen": int((mask & blocked).sum())})
        for key, predictions_at_one in prediction["robust"].items():
            accuracy = prediction_accuracy(predictions_at_one, y_seen)
            robustness.append({"scoring": name, "condition": key, "seen_top1": accuracy,
                               "drop_pp": round(metrics["seen_top1"] - accuracy, 2)})
        for split, targets, count in (("seen_only", y_seen, boundary),
                ("unseen_only", y_unseen, len(pairs) - boundary)):
            restricted.append({"scoring": name, "split": split, "candidates": count,
                               **{f"top{k}": prediction_accuracy(prediction[split][k], targets) for k in (1, 5)}})
    return ({"results": reports, "restricted_candidates": restricted, "robustness": robustness,
             "unseen_ranking_diagnostics": ranking, "unseen_per_pair": per_pair},
            {name: pred["seen"][1].squeeze(1) for name, pred in predictions.items()})


def official_evaluation():
    if DEBUG or winner_run is None:
        raise RuntimeError("Complete the selected model's full refit before official evaluation.")
    recipe = plan["recipe_selected_on_tune"]
    if read_final_lock(recipe) != winner_run:
        raise RuntimeError("The refitted checkpoint differs from the selected lock.")
    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    filenames = tuple(official_filenames(path) for path in paths)
    seen_names, unseen_names = map(set, filenames)
    if seen_names & unseen_names or set(train_full.image) & (seen_names | unseen_names):
        raise ValueError("Training and official split filenames overlap.")
    predictions = predict_recipe(recipe, winner_run, pairs, filenames)
    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], filenames[0], rows, seen=True)
    y_unseen = official_targets(paths[1], filenames[1], rows, seen=False)
    report, predicted_seen = score_trial(winner_run, pairs, predictions, y_seen, y_unseen)
    summary = {"selection": winner_run, "recipe_selected_on_tune": recipe,
               "samples": {"seen": len(y_seen), "unseen": len(y_unseen)},
               "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist()}
    for key, values in report.items():
        summary[key] = [{"recipe": recipe, **row} for row in values]
    winners = {f"{recipe}_{rule}": values for rule, values in predicted_seen.items()}
    return summary, winners, y_seen


evaluation = None if DEBUG else official_evaluation()


# 7. Save

- `development/development_ablation.csv` compares the three recipes on the independent train-derived proxy check.
- `official/official_raw_results.csv` and `official/official_calibrated_results.csv` report the selected model.
- `checkpoints/<run>/<selected_recipe>/model.pt` is the full refitted checkpoint. Its matching lock is under `final/<selected_recipe>/selection_lock.json`.

In [ ]:
def save_results(summary, seen_winners, seen_targets):
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    # Align the team confusion plot order only after predictions and metrics are fixed.
    pairs = [tuple(pair) for pair in summary["candidate_pairs"]]
    lookup = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if set(pairs) != lookup.keys():
        raise ValueError("Candidate identities differ from the common evaluator.")
    to_common = torch.tensor([lookup[pair] for pair in pairs])
    canonical_targets = to_common[seen_targets]
    if (canonical_targets >= len(train_pairs)).any():
        raise ValueError("The common confusion plot expects seen target rows.")
    for name, winners in seen_winners.items():
        votes = F.one_hot(to_common[winners], num_classes=len(pairs)).float()
        stem = (OFFICIAL_DIR / name).relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(votes, canonical_targets, stem, final_evaluation=True)

    table = pd.DataFrame(summary["results"])
    table.to_csv(OFFICIAL_DIR / "official_comparison.csv", index=False)
    for mode, part in table.groupby("scoring", sort=False):
        part.to_csv(OFFICIAL_DIR / f"official_{mode}_results.csv", index=False)
    for name in ("restricted_candidates", "robustness", "unseen_ranking_diagnostics", "unseen_per_pair"):
        pd.DataFrame(summary[name]).to_csv(OFFICIAL_DIR / f"{name}.csv", index=False)
    for row in summary["results"]:
        atomic_json(OFFICIAL_DIR / f"{row['recipe']}_{row['scoring']}.json", row)
    atomic_json(OFFICIAL_DIR / "official_report.json", summary)
    return table


if evaluation is not None:
    table = save_results(*evaluation)
    columns = ["recipe", "feature_swapping", "gamma", "epoch", "selected_on_tune",
               "seen_top1", "unseen_top1", "unseen_top5", "overall_top1", "harmonic_mean"]
    for mode, part in table.groupby("scoring", sort=False):
        print(f"{mode.capitalize()} results")
        display(part[columns])
    recipe = plan["recipe_selected_on_tune"]
    print(recipe, "checkpoint:", (CHECKPOINT_DIR / recipe / "model.pt").resolve())
    print("Official results:", OFFICIAL_DIR.resolve())
else:
    print("Debug logs:", DEVELOPMENT_DIR.resolve())
